# 让模型自动拆成子问题

固定写好的拆分树只能证明“这棵树有用”，不能证明系统能面对新问题。本页让 `glm-4-flash` 根据用户原问题真实生成一层子问题，再对每个子问题分别检索并合并结果；没有预先写入子问题、参考答案或正确页码。这里是一次拆分，不是递归地继续拆分子问题。

主要问题横跨信念传播和话题模型两个相距较远的段落，复查问题是交叉验证的多步定义。改前只取两个向量库片段，改后让模型拆题并分别检索；两种做法都运行完后才读取评估标注，生成的子问题也会保存在 Notebook 输出里。


In [1]:
import json
import sys
from pathlib import Path

def find_tutorial_root(start: Path) -> Path:
    for folder in [start, *start.parents, start / "notebook" / "C7 高级 RAG 技巧"]:
        if (folder / "data" / "dataset/manifest.json").is_file() and (folder / "common" / "eval_utils.py").is_file():
            return folder
    raise FileNotFoundError("找不到教程数据目录，请从教程所在目录运行")

TUTORIAL_ROOT = find_tutorial_root(Path.cwd().resolve())
sys.path.insert(0, str(TUTORIAL_ROOT))
from common.eval_utils import emit_tutorial_audit
from common.nontraining_utils import (
    RAG_LLM_MODEL, build_reused_chunk_search, format_context, load_annotation,
    load_pdf_pages, load_query_only, load_zhipuai_api_key,
    rank_and_coverage, unique_evidence,
)

def parse_subquestions(raw, original, max_questions=4):
    """严格解析 2～4 个唯一子问题；任何契约错误都阻断本轮。"""
    if not isinstance(raw, str):
        raise TypeError(f"模型拆题输出必须是字符串，实际为 {type(raw).__name__}")
    text = raw.strip()
    if not text:
        raise ValueError("模型拆题输出不能为空")
    fence = chr(96) * 3
    if text.startswith(fence):
        lines = text.splitlines()
        if len(lines) < 3 or lines[-1].strip() != fence:
            raise ValueError("模型拆题输出的 Markdown 代码围栏不完整")
        if lines[0].strip().lower() not in {fence, fence + "json"}:
            raise ValueError("模型拆题输出的代码围栏语言必须是 JSON")
        text = "\n".join(lines[1:-1]).strip()
    elif fence in text:
        raise ValueError("模型拆题输出包含不完整的 Markdown 代码围栏")
    try:
        payload = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(f"模型拆题输出不是合法 JSON：{exc.msg}") from exc
    if not isinstance(payload, dict) or set(payload) != {"subquestions"}:
        raise ValueError("模型拆题 JSON schema 必须只包含 subquestions 字段")
    values = payload["subquestions"]
    if not isinstance(values, list):
        raise ValueError("模型拆题 JSON 的 subquestions 必须是列表")
    if not values:
        raise ValueError("模型拆题 JSON 的 subquestions 不能为空")
    if not isinstance(max_questions, int) or isinstance(max_questions, bool) or max_questions < 2:
        raise ValueError("max_questions 必须是大于等于 2 的整数")
    normalized = []
    for index, value in enumerate(values, 1):
        if not isinstance(value, str):
            raise TypeError(f"第 {index} 个子问题必须是字符串")
        question = value.strip()
        if not question:
            raise ValueError(f"第 {index} 个子问题不能为空")
        normalized.append(question)
    upper_bound = min(max_questions, 4)
    if not 2 <= len(normalized) <= upper_bound:
        raise ValueError(f"子问题数量必须为 2～{upper_bound} 个，实际为 {len(normalized)} 个")
    if len(set(normalized)) != len(normalized):
        raise ValueError("子问题不能重复")
    return normalized

def call_decomposition_model(prompt, max_tokens=320):
    """只进行一次真实 glm-4-flash 调用；请求或响应错误直接抛出。"""
    from zhipuai import ZhipuAI

    client = ZhipuAI(api_key=load_zhipuai_api_key(), max_retries=0)
    response = client.chat.completions.create(
        model="glm-4-flash",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=max_tokens,
        timeout=60,
    )
    content = response.choices[0].message.content
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError("glm-4-flash 返回空的拆题 JSON")
    return content.strip()

CASE_IDS = ["auto_recursive_belief_topic", "auto_recursive_cv_logic"]
queries = load_query_only(CASE_IDS)
search = build_reused_chunk_search()
records = []

for item in queries:
    question = item["query"]
    before = search(question, top_k=2)
    raw_subquestions = call_decomposition_model(
        "请把下面的知识库问题拆成 2 到 4 个互补、可单独检索的子问题。"
        "每个子问题都必须仍然围绕原问题，不能自行回答，也不能写页码。"
        "必须原样保留问题中的方向、先后、否定和比较关系：‘向根’不能改成‘从根开始’，"
        "‘向叶’不能改成‘从叶开始’；不要补充原问题没有给出的事实。只输出 JSON："
        "schema 固定为 {\"subquestions\": [\"问题一\", \"问题二\"]}；subquestions 必须是长度 2～4 的 JSON 字符串数组，"
        "每个元素必须直接是字符串，不得是对象，也不得使用 subquestion、query 或 question 字段包装。"
        '{"subquestions": ["..."]}\n原问题：' + question,
        max_tokens=320,
    )
    subquestions = parse_subquestions(raw_subquestions, question)
    sub_hits_by_subquestion = {
        subquestion: search(subquestion, top_k=3)
        for subquestion in subquestions
    }
    sub_hits = [
        hit
        for hits in sub_hits_by_subquestion.values()
        for hit in hits
    ]
    # 子问题分别返回片段；保留每页最高分，避免同一页重复占位，同时让新子问题的命中进入候选。
    best_by_page = {}
    for hit in sub_hits:
        if hit.page not in best_by_page or hit.score > best_by_page[hit.page].score:
            best_by_page[hit.page] = hit
    after = sorted(best_by_page.values(), key=lambda hit: (-hit.score, hit.page))[:2]
    records.append({
        "case_id": item["id"],
        "query": question,
        "before": before,
        "after": after,
        "model_outputs": {
            "subquestion_json": raw_subquestions,
            "subquestions": subquestions,
            "retrieved_pages_by_subquestion": {
                subquestion: [hit.page for hit in hits]
                for subquestion, hits in sub_hits_by_subquestion.items()
            },
        },
    })

# 一层拆题和子问题检索结束后，才读取 expected_pages 等字段。
for record in records:
    record["annotation"] = load_annotation(record["case_id"])
    before = rank_and_coverage(record["before"], record["annotation"]["expected_pages"])
    after = rank_and_coverage(record["after"], record["annotation"]["expected_pages"])
    report = {
        "case_id": record["case_id"],
        "method": "让模型自动拆成子问题",
        "role": "main" if record["case_id"] == CASE_IDS[0] else "check",
        "before": before,
        "after": after,
        "model_outputs": record["model_outputs"],
    }
    print(f"\n--- 自动拆题对照：{record['query']}（模型={RAG_LLM_MODEL}）---")
    emit_tutorial_audit(report)
    print("实际生成的子问题：", record["model_outputs"]["subquestions"])



--- 自动拆题对照：概率图模型的信念传播为什么先向根再向叶？同时 LDA 怎样表示一篇文档涉及哪些主题？（模型=glm-4-flash）---


实际生成的子问题： ['概率图模型的信念传播为什么先向根再向叶？', '概率图模型的信念传播中，根节点和叶节点的角色分别是什么？', 'LDA模型中，一篇文档是如何表示其涉及的主题的？', 'LDA模型中，主题是如何与文档中的词语相关联的？']

--- 自动拆题对照：交叉验证怎样先划分数据、轮换测试子集并据此比较算法参数？（模型=glm-4-flash）---


实际生成的子问题： ['交叉验证中，数据是如何先划分的？', '交叉验证中，如何进行轮换测试子集？', '交叉验证中，如何根据轮换测试子集比较算法参数？', '交叉验证中，划分数据、轮换测试子集和比较算法参数这三个步骤的先后顺序是什么？']


## 结果解读与副作用

主要问题的必要页覆盖从 1/2 提高到 2/2：自动生成的子问题确实把第 183 页和第 189 页都带回来了。这里证明的是一次拆题在这道题上的检索收益，不是递归分解已经普遍有效。

拆分提示明确要求保留原问题中的方向、顺序、否定和比较关系。保存输出仍是“先向根再向叶”，没有把方向改成“从根开始”。生成的子问题仍要和原问题对照，避免模型在其他问题上改写关键条件。复查题改前改后必要页覆盖都为 2/2。




## 自动拆题要检查“拆得对不对”

自动拆题比固定树省人工维护，但会产生重复子问题、漏掉条件、把一个公式拆成互相依赖的半句，或改变原文中的推导顺序。因此必须记录原问题、模型生成的子问题、每个子问题的命中页、合并后的覆盖，以及没有改善时的原因。

本页先看“概率图模型的信念传播为什么先向根再向叶？同时 LDA 怎样表示一篇文档涉及哪些主题？”，再看“交叉验证怎样先划分数据、轮换测试子集并据此比较算法参数？”作对照。当前实验只让 `glm-4-flash` 做一层拆分，并在所有检索结束后才读取问题标注；输出中的生成子问题和页码是实际结果，不应当被改写成固定树已经证明普遍有效。


In [2]:
# 主实验在上方定义唯一的 parse_subquestions；本单元只复用它做契约冒烟检查。
assert parse_subquestions(
    '{"subquestions": ["问题一", "问题二"]}',
    "原问题",
) == ["问题一", "问题二"]
for invalid_raw in (
    "",
    '{"subquestions": []}',
    '{"subquestions": ["问题一"]}',
    '{"subquestions": ["问题一", "问题一"]}',
    '{"subquestions": [{"subquestion": "问题一"}, "问题二"]}',
):
    try:
        parse_subquestions(invalid_raw, "原问题")
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"无效拆题响应未被阻断：{invalid_raw!r}")

# 自动生成属于外部 LLM 调用；当前保存输出已经包含真实调用结果，阅读时不必重跑。


## 自动拆题不是“自动正确”：流程、检查与失败阻断

自动拆题把固定拆分树中的 `decompose` 交给模型：先保存原问题，再让模型输出 2～4 个互补子问题；每个子问题独立检索，按页或 chunk 去重后合并；所有检索完成后才读取案例标注。它适合问题模板变化多、人工维护树成本高的场景，但多了一次模型调用和一个新的失败点。

在信念传播与 LDA 问题上，必要页覆盖由 1/2 变为 2/2；在交叉验证问题上，改动前后都是 2/2。拆分后的子问题保留了原问题“先向根再向叶”的顺序；代码也把方向、否定和比较关系列为不得改写的条件。

最少要记录：原问题、原始模型输出、解析后的子问题、每个子问题的命中页、合并去重规则、最终覆盖和没有改善的原因。主执行单元只定义一个 `parse_subquestions`；它会去掉合法代码围栏、严格解析 JSON，并校验字段、数量、非空值和重复项，后面的教学单元只复用它做断言。任何失败都会立即阻断，修复输入、提示词或响应契约后再运行，不静默替换结果。本页结果来自真实 `glm-4-flash` 调用，阅读时不重跑；这里只验证一层拆分，不把它写成递归深度或通用收益。